# Step 2: Create Features (ATR)
This notebook calculates the Average True Range (ATR) for every stock in our tradable universe.

**Workflow:**
1. Load the raw OHLCV data (`top_5000_yf_data.pkl`)
2. Load the universe mask (`stores/universe_5m.parquet`) created by notebook 1
3. Calculate ATR using the vectorized function from `technical_indicators.py`
4. Mask the ATR so only in-universe stocks have values (out-of-universe → NaN)
5. Save to `stores/features.parquet` for downstream use

In [1]:
import pandas as pd
import numpy as np
import os
import sys

sys.path.append(os.path.abspath('phase2_qrt_challenge/scripts'))
from technical_indicators import calculate_atr_vectorized

import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "stores"
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
# Load raw OHLCV data
print('Loading top_5000_yf_data.pkl...')
pv = pd.read_pickle('top_5000_yf_data.pkl')
print(f'Data loaded. Shape: {pv.shape}')
print(f'Date range: {pv.index[0].date()} → {pv.index[-1].date()}')
print(f'Tickers: {len(pv["Close"].columns)}')

Loading top_5000_yf_data.pkl...
Data loaded. Shape: (4112, 30012)
Date range: 2010-01-04 → 2026-05-08
Tickers: 5002


In [3]:
# Load the tradable universe mask from notebook 1
universe_path = os.path.join(DATA_DIR, 'universe_5m.parquet')
df_universe = pd.read_parquet(universe_path)
print(f'Universe mask loaded. Shape: {df_universe.shape}')
print(f'Avg tradable stocks per day: {df_universe.sum(axis=1).mean():.0f}')

Universe mask loaded. Shape: (4112, 4999)
Avg tradable stocks per day: 1817


In [4]:
# Calculate ATR using vectorized function (runs in seconds)
print('Calculating ATR...')
indicators_dict = calculate_atr_vectorized(pv)
atr_all = indicators_dict['average_true_range']
print(f'Raw ATR shape: {atr_all.shape}')

Calculating ATR...
Raw ATR shape: (4112, 5002)


In [5]:
# Align ATR with the universe mask
# Only keep ATR values for stocks that are in the tradable universe on each day
# Out-of-universe stocks get NaN
common_dates = atr_all.index.intersection(df_universe.index)
common_tickers = atr_all.columns.intersection(df_universe.columns)

atr_aligned = atr_all.loc[common_dates, common_tickers]
universe_aligned = df_universe.loc[common_dates, common_tickers]

# Apply the mask: where universe == 0, set ATR to NaN
atr_masked = atr_aligned.where(universe_aligned == 1)

print(f'Masked ATR shape: {atr_masked.shape}')
print(f'Avg stocks with valid ATR per day: {atr_masked.notna().sum(axis=1).mean():.0f}')

Masked ATR shape: (4112, 5002)
Avg stocks with valid ATR per day: 1812


In [6]:
# Downcast to float32 to save memory
atr_masked = atr_masked.astype('float32')

# Preview
print('\nSample ATR values (last 5 days, first 10 in-universe stocks):')
last_day = atr_masked.iloc[-1]
valid_tickers = last_day.dropna().index[:10].tolist()
atr_masked[valid_tickers].tail()


Sample ATR values (last 5 days, first 10 in-universe stocks):


Ticker,A,AA,AAL,AAMI,AAOI,AAON,AAP,AAPL,AAT,AAUC
Date,,,,,,,,,,
2026-05-04,3.529287,2.620071,0.547857,2.576071,18.653427,5.308572,2.257143,6.844284,0.408572,0.741000
2026-05-05,3.848572,2.620071,0.567143,2.507499,18.985928,5.285716,2.238571,6.795713,0.420000,0.812072
2026-05-06,3.819286,2.565786,0.582857,2.517857,18.818071,5.530715,2.275000,6.872140,0.427857,0.852786
2026-05-07,3.705001,2.218643,0.531429,2.464285,20.059141,8.670715,2.350000,6.689998,0.415714,0.930643
2026-05-08,3.962858,2.169500,0.499286,2.586786,21.656284,9.106430,2.430714,6.923571,0.422857,0.961357


In [10]:
import pyarrow as pa
import pyarrow.parquet as pq

# Build and flatten MultiIndex columns
features_df = pd.concat({'average_true_range': atr_masked}, axis=1)
features_df = features_df.astype('float32')
features_df.columns = [f"{feat}_{ticker}" for feat, ticker in features_df.columns]

# Remove duplicate columns (caused by NaN ticker symbols)
features_df = features_df.loc[:, ~features_df.columns.duplicated()]

output_path = os.path.join(DATA_DIR, 'features.parquet')

table = pa.Table.from_pandas(features_df.reset_index(), preserve_index=False)
pq.write_table(table, output_path, compression='zstd')

print(f'Saved to {output_path}')
print(f'Final features shape: {features_df.shape}')
print(f'Memory usage: {features_df.memory_usage(deep=True).sum() / 1e6:.1f} MB')


Saved to stores/features.parquet
Final features shape: (4112, 4999)
Memory usage: 82.3 MB
